In [ ]:
# 사전 설치 라이브러리 : pip install gradio langchain-google-genai ultralytics pillow pandas requests
# 사전 api 키 발급 : 구글 AI Studio(https://aistudio.google.com/), kakao developers(https://developers.kakao.com/)
import os
import gradio as gr
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
import requests
from urllib.parse import quote  # 퍼센트 인코딩이라는 방식으로 한글 변환 (ex. %EC%95%88%EB%85%95%ED%95%98%EC%84%B8%EC%9A%94)
from ultralytics import YOLO
import base64  # 바이너리 데이터를 64개의 ASCII 문자(A-Z, a-z, 0-9, +, /)로 표현하는 인코딩 방식
from io import BytesIO   # 바이너리 데이터를 메모리에서 조작, 임시 저장 할 때 사용
from PIL import Image
import pandas as pd

In [ ]:
# 환경 변수 설정
os.environ["GOOGLE_API_KEY"] = "Google ai studio에서 발급받은 API 키를 넣으세요"  # Google AI Studio API 키
os.environ["KAKAO_API_KEY"] = "Kakao developers에서 발급받은 API 키를 넣으세요"    # Kakao REST API 키

In [ ]:
# Gemini API 및 YOLO 모델 설정
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
yolo_model = YOLO("yolov8n.pt")

In [ ]:
def analyze_animal_image(image):
    # YOLO로 동물 탐지
    results = yolo_model(image)
    detected_objects = [results[0].names[int(cls)] for cls in results[0].boxes.cls] # 탐지된 객체들의 번호(Index) : 16번은 개, 17번은 고양이

    # 이미지를 base64로 변환
    buffered = BytesIO()  # 메모리 상에서 파일처럼 다룰 수 있는 버퍼
    image.save(buffered, format="PNG")
    encoded_image = base64.b64encode(buffered.getvalue()).decode('utf-8')

    # Gemini API로 종 및 품종 분석 (한글 프롬프트)
    message = HumanMessage(content=[
        {"type": "text", "text": f"탐지된 객체: {', '.join(detected_objects)}. 이 동물의 종과 품종을 식별해 주세요."},
        {"type": "image_url", "image_url": f"data:image/png;base64,{encoded_image}"}
    ])
    response = llm.invoke([message])

    return response.content, results[0].plot()

In [ ]:
# 보호소 및 병원 정보 검색 함수 (개별 키워드 검색 후 결과 병합으로 안정성 강화)
def get_shelter_hospital_info(location):
    kakao_api_key = os.environ.get("KAKAO_API_KEY")
    if not kakao_api_key or "YOUR_KAKAO_API_KEY" in kakao_api_key:
        return "카카오 API 키를 설정해주세요. 환경 변수 KAKAO_API_KEY가 유효하지 않습니다.", pd.DataFrame()

    if not location:
        return "위치를 입력해주세요.", pd.DataFrame()

    # --- 1. 입력된 지역명을 좌표로 변환 ---
    addr_to_coord_url = "https://dapi.kakao.com/v2/local/search/address.json"  # 주소 검색 기능에 대한 카카오 웹주소
    headers = {"Authorization": f"KakaoAK {kakao_api_key}"}  # 인증헤더
    params = {"query": location} # 검색할 주소
    try:
        response = requests.get(addr_to_coord_url, headers=headers, params=params)
        response.raise_for_status()  # 200(정상통과), 400/401/403/500(예외발생)
        address_results = response.json().get("documents", [])

        if not address_results:
            keyword_to_coord_url = "https://dapi.kakao.com/v2/local/search/keyword.json"
            response = requests.get(keyword_to_coord_url, headers=headers, params=params)
            response.raise_for_status()
            address_results = response.json().get("documents", [])

        if not address_results:
            return f"'{location}'에 대한 위치 정보를 찾을 수 없습니다. 좀 더 구체적인 장소나 주소를 입력해보세요.", pd.DataFrame()

        coord_x = address_results[0]['x']  # x(경도): 가장 정확도(유사도)가 높은 첫 번째 결과
        coord_y = address_results[0]['y']  # y(위도): 가장 정확도(유사도)가 높은 첫 번째 결과

    except requests.exceptions.RequestException as e:
        return f"API 요청 중 오류가 발생했습니다(좌표 변환): {e}", pd.DataFrame()

    # --- 2. '동물병원', '동물보호소'를 각각 검색 후 결과 합치기 ---
    queries = ["동물병원", "동물보호소"]
    all_results = {} # 검색결과공간: 중복 제거를 위해 딕셔너리 사용 (key: 장소 ID, alue: 장소 정보 전체)

    for query in queries:
        keyword_search_url = "https://dapi.kakao.com/v2/local/search/keyword.json"
        params = {
            "query": query,
            "x": coord_x,
            "y": coord_y,
            "radius": 10000, # 10km 반경
            "size": 15,  # 한 번의 요청에서 가져올 최대 결과 수
            "sort": "distance"  # 기준 좌표로부터 가까운 순서대로 정렬
        }

        try:
            response = requests.get(keyword_search_url, headers=headers, params=params)
            response.raise_for_status()
            search_results = response.json().get("documents", [])

            # 검색 결과를 장소고유 ID를 키로 하여 딕셔너리에 추가 (중복 자동 제거)
            for result in search_results:
                all_results[result['id']] = result

        except requests.exceptions.RequestException as e:
            # 한쪽 검색이 실패해도 계속 진행하도록 경고만 출력
            print(f"Warning: '{query}' 검색 중 오류 발생 - {e}")

    # --- 3. 최종 결과 처리 ---
    if not all_results:
        return f"'{location}' 주변 10km 내에 동물병원이나 보호소 정보가 없습니다.", pd.DataFrame()

    # 딕셔너리의 값들을 리스트로 변환
    final_list = list(all_results.values())  # 장소 key는 필요없고 장소 정보(value)만 필요

    # 거리순으로 다시 정렬 (중요!)
    # API 결과의 distance는 문자열일 수 있으므로 int로 변환하여 정렬
    final_list.sort(key=lambda x: int(x['distance']))

    # DataFrame 생성
    data = []
    for result in final_list:
        category = "동물병원" if "동물병원" in result.get("category_name", "") else "보호소/기타"
        data.append({
            "이름": result.get("place_name", "정보 없음"),
            "주소": result.get("road_address_name") or result.get("address_name", "정보 없음"),
            "연락처": result.get("phone", "정보 없음"),
            "유형": category
        })

    df = pd.DataFrame(data)

    # Gemini API 요약 요청
    summary_prompt = f"'{location}' 주변의 동물병원 및 보호소 목록입니다. 이 목록을 바탕으로 사용자에게 어떤 장소들이 있는지 간략히 요약해주세요.\n\n{df.to_string()}"
    message = HumanMessage(content=summary_prompt)
    summary = llm.invoke([message]).content

    return summary, df

In [ ]:
# 동물 증상 진단 함수
def diagnose_animal_symptoms(symptoms):
    message = HumanMessage(content=f"내 반려동물이 다음 증상을 보입니다: {symptoms}. 가능한 원인과 조언을 수의사처럼 제공해 주세요. 단, 이는 전문적인 진단이 아니며, 긴급 상황에서는 동물병원 방문을 권장한다고 명시해 주세요.")
    response = llm.invoke([message])
    return response.content

In [ ]:
# 수의사 챗봇과 자유 대화 함수
def vet_expert_chat(user_input, history):
    # 대화 기록을 포함
    system_prompt = SystemMessage(content="당신은 수의사 전문가입니다. 반려동물의 건강, 훈련, 영양, 행동 등에 대해 전문적이고 친절한 조언을 한글로 제공하세요. 비전문가적 조언임을 명시하고, 필요 시 동물병원 방문을 권장하세요.")
    messages = [system_prompt] + [HumanMessage(content=msg) for msg in history] + [HumanMessage(content=user_input)]
    response = llm.invoke(messages)

    # 대화 기록 업데이트
    history.append(user_input)
    history.append(response.content)
    return response.content, history

In [ ]:
with gr.Blocks(title="YOLO + Gemini + Kakao 챗봇") as demo:
    gr.Markdown("# 동물 챗봇 서비스")

    with gr.Tab("동물 이미지 분석"):
        image_input = gr.Image(type="pil", label="동물 이미지를 업로드하세요")
        analyze_button = gr.Button("분석하기")
        image_output = gr.Textbox(label="분석 결과", lines=6)
        annotated_image = gr.Image(label="탐지analyze_animal_ima된 동물 이미지")
        analyze_button.click(ge, inputs=image_input, outputs=[image_output, annotated_image])

    with gr.Tab("보호소 및 병원 정보"):
        location_input = gr.Textbox(label="위치(예: 서울 강남구)를 입력하세요")
        search_button = gr.Button("검색하기")
        # Textbox는 요약 정보를 표시하도록 명확히 함
        summary_output = gr.Textbox(label="정보 요약", lines=15)
        # HTML 출력 대신 DataFrame 컴포넌트로 변경하여 테이블 형태로 출력
        dataframe_output = gr.DataFrame(
            label="보호소/병원 목록",
            headers=["이름", "주소", "연락처", "유형"],
            datatype=["str", "str", "str", "str"],
            row_count=5, # 한 번에 보여줄 행의 수
            col_count=(4, "fixed") # 열의 수 고정
        )
        # outputs를 수정된 컴포넌트에 맞게 변경
        search_button.click(
            get_shelter_hospital_info,
            inputs=location_input,
            outputs=[summary_output, dataframe_output]
        )

    with gr.Tab("동물 증상 진단"):
        symptoms_input = gr.Textbox(label="동물의 증상을 입력하세요")
        diagnose_button = gr.Button("진단하기")
        diagnosis_output = gr.Textbox(label="진단 조언", lines=30)
        diagnose_button.click(diagnose_animal_symptoms, inputs=symptoms_input, outputs=diagnosis_output)

    with gr.Tab("수의사와 자유 대화"):
        chatbot_input = gr.Textbox(label="질문을 입력하세요 (예: 강아지 훈련 방법)")
        chatbot_output = gr.Textbox(label="수의사 조언", lines=30)
        chat_history = gr.State(value=[])
        chat_button = gr.Button("전송")
        chat_button.click(vet_expert_chat, inputs=[chatbot_input, chat_history], outputs=[chatbot_output, chat_history])

demo.launch()

In [ ]:
demo.close()